# 06 — SLA & Trend Analytics
**SuaraLens** | Prototipe analytics layer — murni agregasi data, tanpa model AI.
Fungsi-fungsi di sini akan dipindahkan ke SQL view / query Postgres di backend.


In [ ]:
import sys
sys.path.insert(0, '..')

import json
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path
from modules.analytics import (
    compute_sla_breach_by_category,
    compute_sla_breach_by_urgency,
    compute_avg_resolution_time,
    compute_monthly_trend,
    detect_trend_anomalies,
    compute_stakeholder_monthly,
    build_sla_analytics_payload,
    build_trend_analytics_payload,
    save_json,
)

sns.set_theme(style='whitegrid')
DATA_PATH = '../data/suaralens_dummy_simulasi.jsonl'

df = pd.read_json(DATA_PATH, lines=True)
print(f'Dataset: {len(df):,} baris')


## 1. SLA Breach per Kategori

In [ ]:
sla_by_cat = compute_sla_breach_by_category(df)
df_sla_cat = pd.DataFrame(sla_by_cat)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(df_sla_cat['kategori'], df_sla_cat['breach_pct'],
               color=sns.color_palette('RdYlGn_r', len(df_sla_cat)))
ax.set_xlabel('SLA Breach (%)')
ax.set_title('Persentase SLA Breach per Kategori')
ax.axvline(df_sla_cat['breach_pct'].mean(), color='navy', linestyle='--', alpha=0.7,
           label=f'Rata-rata: {df_sla_cat["breach_pct"].mean():.1f}%')
ax.legend()
for bar, row in zip(bars, df_sla_cat.itertuples()):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            f'{row.breach_pct:.1f}%', va='center', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 2. SLA Breach per Urgency & Rata-rata Waktu Penyelesaian

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SLA breach per urgency
sla_by_urg = compute_sla_breach_by_urgency(df)
df_sla_urg = pd.DataFrame(sla_by_urg)
colors_urg = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
axes[0].bar(df_sla_urg['urgency'], df_sla_urg['breach_pct'], color=colors_urg)
axes[0].set_title('SLA Breach per Urgency Level')
axes[0].set_ylabel('Breach (%)')
for i, row in df_sla_urg.iterrows():
    axes[0].text(i, row['breach_pct'] + 0.5, f'{row["breach_pct"]:.1f}%', ha='center')

# Rata-rata resolusi
avg_res = compute_avg_resolution_time(df)
df_avg  = pd.DataFrame(avg_res)
axes[1].barh(df_avg['kategori'], df_avg['avg_days'],
             color=sns.color_palette('Blues_r', len(df_avg)))
axes[1].set_title('Rata-rata Hari Penyelesaian per Kategori')
axes[1].set_xlabel('Hari')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()


## 3. Tren Bulanan (Top 5 Kategori)

In [ ]:
trend_data = compute_monthly_trend(df, top_n=5)
df_trend   = pd.DataFrame(trend_data)

fig, ax = plt.subplots(figsize=(13, 5))
for cat in df_trend['kategori'].unique():
    sub = df_trend[df_trend['kategori'] == cat]
    ax.plot(sub['bulan'], sub['jumlah'], marker='o', label=cat, linewidth=2)

ax.set_title('Tren Jumlah Masukan per Bulan (Top 5 Kategori)')
ax.set_xlabel('Bulan')
ax.set_ylabel('Jumlah Masukan')
ax.legend(loc='upper left', fontsize=9)
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


## 4. Deteksi Lonjakan (Anomali)

In [ ]:
anomalies = detect_trend_anomalies(df)
df_anom   = pd.DataFrame(anomalies)
df_anom_flagged = df_anom[df_anom['is_anomaly'] == True]

print(f'Total anomali terdeteksi: {len(df_anom_flagged)} bulan-kategori')
print()
display(df_anom_flagged.sort_values('jumlah', ascending=False).head(15))


## 5. Stakeholder per Bulan

In [ ]:
sh_monthly  = compute_stakeholder_monthly(df)
df_sh       = pd.DataFrame(sh_monthly)

fig, ax = plt.subplots(figsize=(13, 5))
for sh in df_sh['stakeholder_type'].unique():
    sub = df_sh[df_sh['stakeholder_type'] == sh]
    ax.plot(sub['bulan'], sub['jumlah'], marker='s', label=sh, linewidth=2)

ax.set_title('Jumlah Masukan per Stakeholder per Bulan')
ax.set_xlabel('Bulan')
ax.set_ylabel('Jumlah')
ax.legend()
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()


## 6. Simpan Payload JSON

In [ ]:
sla_payload   = build_sla_analytics_payload(df)
trend_payload = build_trend_analytics_payload(df, top_n=5)

save_json(sla_payload,   '../data/output/sla_analytics.json')
save_json(trend_payload, '../data/output/trend_analytics.json')

print('\n✓ Semua output analitik tersimpan di data/output/')


## Insight Utama

1. **SLA breach tidak merata** — kategori tertentu jauh lebih sering melebihi target; prioritaskan perbaikan di sana
2. **Urgency tinggi = breach lebih sering** — tapi tidak selalu; ada kategori Low yang breach-nya tinggi karena volume besar
3. **Pola musiman terlihat** — Keuangan melonjak awal semester, Akademik melonjak saat masa ujian
4. **Mahasiswa mendominasi hampir semua bulan** — wajar untuk platform kampus; perlu pastikan kanal untuk Orang Tua/Mitra juga terjangkau
5. **Waktu resolusi bervariasi jauh antar kategori** — gap ini penting untuk dievaluasi dalam kebijakan SLA

> Fungsi agregasi di `modules/analytics.py` siap dipindahkan ke SQL view/query Postgres untuk backend produksi.
